# dynSystemSim.ipynb

Notebook for single-run simulation (Tab 1 + Tab 2):

- integration (RK45 / DOP853 / RK4 / symplectic)
- time series (Tab 2)
- phase portrait (Tab 1)
- Lyapunov spectrum (Tab 1)

Driven by a StaticParamsConfig.json export from the Streamlit app.


In [ ]:
# --- Imports + project root wiring ---
import json
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / "core").is_dir() and (candidate / "plotting").is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate repo root (missing core/ and plotting/)")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.solver import integrate_system, integrate_system_rk4
from core.symplectic_solver import (
    integrate_system_symplectic_verlet,
    integrate_system_symplectic_fr,
)
from core.lyapunov import compute_lyapunov_spectrum
from core.lorenz_system_rhs import lorenz_rhs
from core.rossler_system_rhs import rossler_rhs
from core.henon_heiles_system_rhs import (
    henon_heiles_rhs,
    henon_heiles_dq_dt,
    henon_heiles_dp_dt,
)
from core.jacobians_fixed_systems import lorenz_jac, rossler_jac, henon_heiles_jac
from plotting.plotting import phase_portrait


## 1) Load config (StaticParamsConfig.json)

Place your exported `StaticParamsConfig.json` next to this notebook,
or update the path below.


In [ ]:
# --- Load config (fallback to a default example if missing) ---
assert PROJECT_ROOT is not None

CONFIG_CANDIDATES = [
    Path("StaticParamsConfig.json"),
    Path("StaticParamsConfig(4).json"),
    PROJECT_ROOT / "examples" / "StaticParamsConfig.json",
]


def load_config(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


cfg = None
for path in CONFIG_CANDIDATES:
    if path.exists():
        cfg = load_config(path)
        print(f"Loaded config: {path.resolve()}")
        break

if cfg is None:
    cfg = {
        "schema_version": "1.0",
        "system": {
            "system_key": "lorenz",
            "var_names": ["x", "y", "z"],
            "eq_lines": [],
            "params": {"sigma": 10.0, "rho": 28.0, "beta": 8.0 / 3.0},
        },
        "integration": {
            "t0": 0.0,
            "tf": 50.0,
            "dt": 0.01,
            "y0": [1.0, 1.0, 1.0],
            "solver_kind": "rk45",
            "solve_options": {"rtol": 1e-6, "atol": 1e-8},
        },
        "postprocess": {"transient_steps": 0},
        "plots": {
            "plot_mode": "2D phase plane",
            "phase_axes": {"x_idx": 0, "y_idx": 1, "z_idx": None},
            "time_series_indices": [0, 1, 2],
        },
        "lyapunov": {
            "enabled": True,
            "settings": {
                "t_transient": 0.0,
                "t_measure": 10.0,
                "qr_every_steps": 10,
                "jacobian": "analytic",
                "fd_eps": 1e-8,
            },
        },
    }
    print("Loaded fallback config (Lorenz).")


## 2) Build RHS / Jacobian

Supported `system_key`:
- `lorenz`
- `rossler`
- `henon_heiles`
- `custom` (SymPy expressions in `eq_lines`)


In [ ]:
SAFE_FUNCS = {
    "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "exp": sp.exp, "log": sp.log, "sqrt": sp.sqrt,
    "sinh": sp.sinh, "cosh": sp.cosh, "tanh": sp.tanh,
    "abs": sp.Abs,
}


def parse_params_text(text: str) -> dict:
    params = {}
    for line in (text or "").splitlines():
        line = line.strip()
        if not line:
            continue
        if "=" not in line:
            raise ValueError(f"Parameter line must be name=value. Got: {line!r}")
        name, val = line.split("=", 1)
        name = name.strip()
        val = val.strip()
        if name.lower() == "t":
            raise ValueError("Parameter name 't' is reserved.")
        params[name] = float(val)
    return params


def build_custom_rhs(var_names, eq_lines, params):
    n = len(var_names)
    if len(eq_lines) != n:
        raise ValueError(f"Need exactly {n} equations. Got {len(eq_lines)}.")

    t_sym = sp.Symbol("t")
    var_syms = sp.symbols(var_names)
    param_syms = {k: sp.Symbol(k) for k in params.keys()}

    locals_dict = {
        **SAFE_FUNCS,
        "t": t_sym,
        **{name: sym for name, sym in zip(var_names, var_syms)},
        **param_syms,
    }

    exprs = []
    for i, line in enumerate(eq_lines):
        s = (line or "").strip()
        if not s:
            raise ValueError(f"Equation {i+1} is empty.")
        exprs.append(sp.sympify(s, locals=locals_dict))

    args = [t_sym] + list(var_syms) + [param_syms[k] for k in params.keys()]
    f_rhs = sp.lambdify(args, exprs, modules=["numpy"])
    param_values = [float(params[k]) for k in params.keys()]

    def rhs(t, y):
        vals = [float(t)] + list(y) + param_values
        return np.asarray(f_rhs(*vals), dtype=float)

    return rhs


def build_custom_rhs_and_jacobian(var_names, eq_lines, params):
    n = len(var_names)
    if len(eq_lines) != n:
        raise ValueError(f"Need exactly {n} equations. Got {len(eq_lines)}.")

    t_sym = sp.Symbol("t")
    var_syms = sp.symbols(var_names)
    param_syms = {k: sp.Symbol(k) for k in params.keys()}

    locals_dict = {
        **SAFE_FUNCS,
        "t": t_sym,
        **{name: sym for name, sym in zip(var_names, var_syms)},
        **param_syms,
    }

    exprs = []
    for i, line in enumerate(eq_lines):
        s = (line or "").strip()
        if not s:
            raise ValueError(f"Equation {i+1} is empty.")
        exprs.append(sp.sympify(s, locals=locals_dict))

    J = sp.Matrix(exprs).jacobian(var_syms)

    args = [t_sym] + list(var_syms) + [param_syms[k] for k in params.keys()]
    f_rhs = sp.lambdify(args, exprs, modules=["numpy"])
    f_jac = sp.lambdify(args, J, modules=["numpy"])
    param_values = [float(params[k]) for k in params.keys()]

    def rhs(t, y):
        vals = [float(t)] + list(y) + param_values
        return np.asarray(f_rhs(*vals), dtype=float)

    def jac(t, y):
        vals = [float(t)] + list(y) + param_values
        return np.asarray(f_jac(*vals), dtype=float)

    return rhs, jac


def build_custom_symplectic_functions(var_names, eq_lines, params):
    n = len(var_names)
    if n % 2 != 0:
        raise ValueError("Symplectic solvers require an even number of variables.")
    if len(eq_lines) != n:
        raise ValueError(f"Need exactly {n} equations. Got {len(eq_lines)}.")

    n_q = n // 2
    t_sym = sp.Symbol("t")
    var_syms = sp.symbols(var_names)
    q_syms = var_syms[:n_q]
    p_syms = var_syms[n_q:]
    param_syms = {k: sp.Symbol(k) for k in params.keys()}

    locals_dict = {
        **SAFE_FUNCS,
        "t": t_sym,
        **{name: sym for name, sym in zip(var_names, var_syms)},
        **param_syms,
    }

    exprs = []
    for i, line in enumerate(eq_lines):
        s = (line or "").strip()
        if not s:
            raise ValueError(f"Equation {i+1} is empty.")
        exprs.append(sp.sympify(s, locals=locals_dict))

    dq_exprs = exprs[:n_q]
    dp_exprs = exprs[n_q:]

    dq_args = [t_sym] + list(p_syms) + [param_syms[k] for k in params.keys()]
    dp_args = [t_sym] + list(q_syms) + [param_syms[k] for k in params.keys()]
    f_dq = sp.lambdify(dq_args, dq_exprs, modules=["numpy"])
    f_dp = sp.lambdify(dp_args, dp_exprs, modules=["numpy"])
    param_values = [float(params[k]) for k in params.keys()]

    def dq_dt(t, p):
        vals = [float(t)] + list(p) + param_values
        return np.asarray(f_dq(*vals), dtype=float)

    def dp_dt(t, q):
        vals = [float(t)] + list(q) + param_values
        return np.asarray(f_dp(*vals), dtype=float)

    return dq_dt, dp_dt


sys_cfg = cfg.get("system", {})
system_key = str(sys_cfg.get("system_key", "lorenz")).lower()
var_names = list(sys_cfg.get("var_names") or [])
eq_lines = list(sys_cfg.get("eq_lines") or [])
params = dict(sys_cfg.get("params") or {})
if not params and sys_cfg.get("params_text"):
    params = parse_params_text(sys_cfg.get("params_text"))

rhs = None
jac = None
dq_dt = None
dp_dt = None

if system_key == "lorenz":
    rhs = lambda t, y: lorenz_rhs(t, y, **params)
    jac = lambda t, y: lorenz_jac(t, y, **params)
    if not var_names:
        var_names = ["x", "y", "z"]
elif system_key == "rossler":
    rhs = lambda t, y: rossler_rhs(t, y, **params)
    jac = lambda t, y: rossler_jac(t, y, **params)
    if not var_names:
        var_names = ["x", "y", "z"]
elif system_key == "henon_heiles":
    rhs = lambda t, y: henon_heiles_rhs(t, y, **params)
    jac = lambda t, y: henon_heiles_jac(t, y, **params)
    dq_dt = lambda t, p: henon_heiles_dq_dt(t, p, **params)
    dp_dt = lambda t, q: henon_heiles_dp_dt(t, q, **params)
    if not var_names:
        var_names = ["q1", "q2", "p1", "p2"]
elif system_key == "custom":
    auto_jac = bool(sys_cfg.get("auto_jacobian", False))
    use_jac = bool(sys_cfg.get("use_jacobian", False))
    if auto_jac and use_jac:
        rhs, jac = build_custom_rhs_and_jacobian(var_names, eq_lines, params)
    else:
        rhs = build_custom_rhs(var_names, eq_lines, params)
        jac = None
else:
    raise ValueError(f"Unknown system_key: {system_key}")


## 3) Integrate the system


In [ ]:
integ = cfg.get("integration", {})
t0 = float(integ.get("t0", 0.0))
tf = float(integ.get("tf", 50.0))
dt = float(integ.get("dt", 0.01))
y0 = np.asarray(integ.get("y0", [1.0] * max(1, len(var_names))), dtype=float)

solver_kind = str(integ.get("solver_kind", "rk45")).lower().strip()
solve_opts: dict[str, object] = dict(integ.get("solve_options") or {})

if solver_kind == "ivp":
    solver_kind = "rk45"

if solver_kind not in ("rk45", "dop853", "rk4", "symplectic_verlet", "symplectic_fr"):
    print(f"Unknown solver_kind {solver_kind!r}; using RK45.")
    solver_kind = "rk45"

if solver_kind == "rk45" and "method" not in solve_opts:
    solve_opts["method"] = "RK45"
elif solver_kind == "dop853" and "method" not in solve_opts:
    solve_opts["method"] = "DOP853"

if solver_kind == "rk4":
    sol = integrate_system_rk4(rhs, t_span=(t0, tf), y0=y0, t_step=dt)
elif solver_kind in ("rk45", "dop853"):
    sol = integrate_system(rhs, t_span=(t0, tf), y0=y0, t_step=dt, **solve_opts)
elif solver_kind in ("symplectic_verlet", "symplectic_fr"):
    if dq_dt is None or dp_dt is None:
        raise ValueError("Symplectic solver selected but dq_dt/dp_dt are not available.")
    if solver_kind == "symplectic_verlet":
        sol = integrate_system_symplectic_verlet(
            rhs,
            t_span=(t0, tf),
            y0=y0,
            t_step=dt,
            dq_dt=dq_dt,
            dp_dt=dp_dt,
        )
    else:
        sol = integrate_system_symplectic_fr(
            rhs,
            t_span=(t0, tf),
            y0=y0,
            t_step=dt,
            dq_dt=dq_dt,
            dp_dt=dp_dt,
        )
else:
    sol = integrate_system(rhs, t_span=(t0, tf), y0=y0, t_step=dt, **solve_opts)

if not sol.success:
    raise RuntimeError(f"Integration failed: {sol.message}")

t = np.asarray(sol.t, dtype=float)
Y = np.asarray(sol.y, dtype=float)
print(f"Integrated: t in [{t[0]:.3f}, {t[-1]:.3f}] with {t.size} steps")


## 4) Tab 2 - Time series (post-transient)


In [ ]:
post = cfg.get("postprocess") or {}
transient_steps = int(post.get("transient_steps", 0))
transient_steps = max(0, min(transient_steps, Y.shape[1] - 1))

t_plot = t[transient_steps:]
Y_plot = Y[:, transient_steps:]

plots_cfg = cfg.get("plots") or {}
ts_indices = plots_cfg.get("time_series_indices") or list(range(Y.shape[0]))

# sanitize indices
clean_indices = []
for i in ts_indices:
    try:
        ii = int(i)
        if 0 <= ii < Y.shape[0]:
            clean_indices.append(ii)
    except Exception:
        continue
if not clean_indices:
    clean_indices = list(range(Y.shape[0]))

fig, ax = plt.subplots(figsize=(8.5, 3.2))
fig.set_dpi(140)
for idx in clean_indices:
    label = var_names[idx] if idx < len(var_names) else f"y{idx}"
    ax.plot(t_plot, Y_plot[idx, :], linewidth=0.9, label=label)
ax.set_xlabel("t")
ax.set_ylabel("value")
ax.grid(True, linewidth=0.3)
ax.legend(loc="best")
plt.show()


## 5) Tab 1 - Phase portrait


In [ ]:
plots_cfg = cfg.get("plots") or {}
axes_cfg = plots_cfg.get("phase_axes") or {"x_idx": 0, "y_idx": 1, "z_idx": None}
xi = int(axes_cfg.get("x_idx", 0))
yi = int(axes_cfg.get("y_idx", 1))
zi = axes_cfg.get("z_idx", None)
plot_mode = str(plots_cfg.get("plot_mode", "2D phase plane"))

if "3d" in plot_mode.lower() and Y.shape[0] >= 3:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    fig = plt.figure(figsize=(7.0, 5.0))
    ax = fig.add_subplot(111, projection="3d")
    z_index = int(zi) if zi is not None else 2
    ax.plot(
        Y_plot[xi, :],
        Y_plot[yi, :],
        Y_plot[z_index, :],
        linewidth=0.6,
    )
    ax.set_xlabel(var_names[xi])
    ax.set_ylabel(var_names[yi])
    ax.set_zlabel(var_names[z_index])
    ax.set_title("Phase portrait (3D)")
    plt.show()
else:
    phase_portrait(
        Y,
        x_index=xi,
        y_index=yi,
        transient_steps=transient_steps,
        xlabel=var_names[xi],
        ylabel=var_names[yi],
        title=f"Phase portrait ({var_names[yi]} vs {var_names[xi]})",
    )


## 6) Tab 1 - Lyapunov spectrum (optional)


In [ ]:
lyap_cfg = cfg.get("lyapunov") or {}
if not bool(lyap_cfg.get("enabled", True)):
    print("Lyapunov disabled in config.")
else:
    s = lyap_cfg.get("settings") or {}
    total_time = float(tf) - float(t0)

    t_transient = s.get("t_transient", None)
    t_measure = s.get("t_measure", None)
    if t_transient is None or t_measure is None:
        transient_steps = s.get("transient_steps", None)
        transient_fraction = s.get("transient_fraction", None)
        if transient_steps is not None:
            t_transient = float(transient_steps) * float(dt)
        elif transient_fraction is not None:
            t_transient = float(transient_fraction) * total_time
        else:
            t_transient = 0.0
        t_measure = total_time - float(t_transient)

    if t_measure <= 0:
        raise ValueError("Not enough time for Lyapunov measurement.")

    qr_every_steps = int(s.get("qr_every_steps", max(1, int(round(0.1 / dt)))))
    fd_eps = float(s.get("fd_eps", 1e-8))

    # For symplectic solvers, use RK45 for Lyapunov (matches app behavior)
    lyap_solver_kind = "rk45" if solver_kind.startswith("symplectic") else solver_kind


    def rhs_wrapped(tt: float, xx: np.ndarray) -> np.ndarray:
        return rhs(tt, xx)

    jac_wrapped: "JacFn | None"
    if jac is None:
        jac_wrapped = None
    else:
        def jac_wrapped(tt: float, xx: np.ndarray) -> np.ndarray:
            return jac(tt, xx)

    res = compute_lyapunov_spectrum(
        rhs=rhs_wrapped,
        x0=np.asarray(y0, dtype=float),
        t0=float(t0),
        dt=float(dt),
        t_transient=float(t_transient),
        t_measure=float(t_measure),
        qr_every_steps=qr_every_steps,
        jac=jac_wrapped,
        fd_eps=fd_eps,
        solver_kind=lyap_solver_kind,
        solve_options=solve_opts,
    )

    print("Lyapunov exponents:")
    print(res.lambdas)
